In [78]:
import pandas as pd
from collections import Counter
from graphviz import Digraph
import os

# 1. Thiết lập thư mục đầu ra
output_dir = "../results/phase_drift_analysis"
os.makedirs(output_dir, exist_ok=True)

# 2. Load và tiền xử lý dữ liệu
df = pd.read_csv("../data/bpi-challenge-2017/bpi_2017_cleaned.csv")
df = df[df["lifecycle:transition"] == "complete"]
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], format='ISO8601')
df['year'] = df['time:timestamp'].dt.year
df['week'] = df['time:timestamp'].dt.isocalendar().week
df = df.sort_values(['case:concept:name', 'time:timestamp'])

def build_dfg(df_sub):
    edges = Counter()
    for _, group in df_sub.groupby('case:concept:name'):
        activities = list(group['concept:name'])
        for i in range(len(activities) - 1):
            edges[(activities[i], activities[i+1])] += 1
    return edges

def draw_dfg(edges, title, file_path, threshold=5):
    dot = Digraph(comment=title)
    dot.attr(rankdir='LR', nodesep='0.4', ranksep='0.8', fontsize='12')
    
    # Lọc cạnh để sơ đồ nhỏ và rõ ràng
    edges = {k: v for k, v in edges.items() if v >= threshold}
    nodes = set([n for e in edges.keys() for n in e])
    
    for n in nodes:
        # Rút gọn tên để sơ đồ không bị quá rộng
        short_name = n.replace('Application', 'App').replace('Accepted', 'Acc')
        dot.node(n, short_name, shape='box', style='rounded,filled', fillcolor='#f9f9f9')
        
    for (src, tgt), freq in edges.items():
        dot.edge(src, tgt, label=str(freq), penwidth=str(0.5 + freq/200))
    
    dot.render(file_path, format='png', cleanup=True)

# 3. ĐỊNH NGHĨA 5 PHASE RỜI NHAU (DISJOINT PHASES)
# Cách chia này giúp so sánh "Trước và Sau" của từng sự kiện một cách tuyệt đối
phases = {
    "Phase_1_W1_18": df[df['week'] < 19],                            # Trước sự kiện 1
    "Phase_2_W19_23": df[(df['week'] >= 19) & (df['week'] < 24)],    # Sau sự kiện 1 (Tuần 19)
    "Phase_3_W24":    df[(df['week'] >= 24) & (df['week'] < 25)],    # Sự kiện bản lề (Tuần 24)
    "Phase_4_W25_48": df[(df['week'] >= 25) & (df['week'] < 49)],    # Sau sự kiện 2 (Tuần 25)
    "Phase_5_W49_End": df[df['week'] >= 49]                          # Cuối năm
}

# 4. Thực thi vẽ sơ đồ
for phase_name, subset in phases.items():
    print(f"Processing {phase_name}...")
    phase_path = os.path.join(output_dir, phase_name)
    os.makedirs(phase_path, exist_ok=True)
    
    for app_type in ['New credit', 'Limit raise']:
        app_label = app_type.replace(' ', '_').lower()
        app_subset = subset[subset['case:ApplicationType'] == app_type]
        
        if not app_subset.empty:
            edges = build_dfg(app_subset)
            # Threshold thấp vì dữ liệu mỗi phase đã bị chia nhỏ, giúp soi kỹ thay đổi
            draw_dfg(edges, f"{phase_name} - {app_type}", 
                     os.path.join(phase_path, f"{app_label}_dfg"),
                     threshold=15)

print(f"\n--- XONG! Sơ đồ đã được lưu tại: {output_dir} ---")

Processing Phase_1_W1_18...
Processing Phase_2_W19_23...
Processing Phase_3_W24...
Processing Phase_4_W25_48...
Processing Phase_5_W49_End...

--- XONG! Sơ đồ đã được lưu tại: ../results/phase_drift_analysis ---
